<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/autosegment_CT_Lung_OARs_SMIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

In this tutorial, we will demonstrate how to apply a pre-trained AI model to segment organs-at-risk (OARs) on a lung CT scan using pyCERR.
Requirements

    Python>=3.8
    Applying this model requires access to a GPU.

## AI model

    The segmentation model was trained and validated on CT scans used for RT planning. It does not work optimally on diagnostic CTs or scans in positions other than Head First Supine.
    The trained model is distributed along with python libraries and other dependencies via a conda package.

## Required input data

    RT planning CT DICOM

## Running the model

The input DICOM data must be located in inputDicomPath. The directory structure is as follows:

    inputDicomPath
        Pat ID 1
        Pat ID 2
        Pat ID 3
        ...  
        
---

## License

By downloading the software you are agreeing to the following terms and conditions as well as to the Terms of Use of CERR software.

THE SOFTWARE IS PROVIDED "AS IS" AND CERR DEVELOPMENT TEAM AND ITS COLLABORATORS DO NOT MAKE ANY WARRANTY, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE, NOR DO THEY ASSUME ANY LIABILITY OR RESPONSIBILITY FOR THE USE OF THIS SOFTWARE.

This software is for research purposes only and has not been approved for clinical use.

Software has not been reviewed or approved by the Food and Drug Administration, and is for non-clinical, IRB-approved Research Use Only. In no event shall data or images generated through the use of the Software be used in the provision of patient care.

You may publish papers and books using results produced using software provided that you reference the appropriate citations:

    SMIT model: https://arxiv.org/abs/2205.10342
    CERR software: https://doi.org/10.1118/1.1568978
    CERR library of model implementations: https://doi.org/10.1016/j.ejmp.2020.04.011

YOU MAY NOT DISTRIBUTE COPIES of this software, or copies of software derived from this software, to others outside your organization without specific prior written permission from the CERR development team except where noted for specific software products.

All technology and technical data delivered under this Agreement are subject to US export control laws and may be subject to export or import regulations in other countries. You agree to comply strictly with all such laws and regulations and acknowledge that you have the responsibility to obtain such licenses to export, re-export, or import as may be required after delivery to you.

In [1]:
import os
import cerr
import subprocess
import numpy as np
import shutil

from pathlib import Path

from cerr import plan_container as pc
from cerr.dcm_export import rtstruct_iod
from cerr.contour import rasterseg as rs
from cerr.contour.rasterseg import getStrMask

from cerr.utils.ai_pipeline import createSessionDir
from cerr.utils.mask import computeBoundingBox

## Specify the working directory

In [2]:
workDir=r'/path/to/model_install_dir'
dcmDir = r'/path/to/input_dicom_dirs'
outputDir r'/path/to/write/segmentations'

modelName='CT_Lung_OAR_SMITplus'

# Directory for temporary files created during inference 
sessionPath = os.path.join(workDir,'session')
os.makedirs(sessionPath, exist_ok=True)

## Download the segmentation model

In [ ]:
%%bash
MODELNUM=12
MODELNAME=CT_Lung_OAR_SMITplus
TOK=#userid:token 

workDir=/path/to/model_install_dir
cd $workDir
cd model_installer
./installer.sh -m ${MODELNUM} -d ${workDir} -p C -u ${TOK}

In [4]:
wrapperInstallDir = os.path.join(modelDir,modelName)
wrapperPath = os.path.join(wrapperInstallDir,'bash_run_SMIT_Segmentation.sh')
weightsPath = os.path.join(wrapperInstallDir,'trained_weights','thorax_ct_oars.pt')

condaEnvDir = r'/path/to/conda_install_dir/envs/CT_Lung_OAR_SMITplus'
condaEnvActivateScript = os.path.join(condaEnvDir, 'bin', 'activate')

In [5]:
# Define label no. to structure map
labelDict = {'Lung_L':1, 'Lung_R':2, 'Esophagus':3, 'Heart':4}
structNames = list(structToLabelMap.keys())

## Run inference

In [ ]:
scanNum = 0
modality = 'CT'

In [ ]:
cwd = os.getcwd()
os.chdir(wrapperInstallDir)

ptList = os.listdir(dcmDir)

# Loop over scans in the input directory
for pt in ptList:
    # Import to planC
    ptDir = os.path.join(dcmDir, pt)
    planC = pc.loadDcmDir(ptDir)

    #Export to NIfTI for inference
    print('Exporting model inputs...')
    modInputPath, modOutputPath = createSessionDir(sessionPath, ptDir)
    basePrefix = os.path.basename(ptDir)
    baseScanFile = os.path.join(modInputPath, basePrefix + '.nii.gz')
    planC.scan[baseScanNum].saveNii(niiFileName=baseScanFile)

    #Run lung OAR Segmentation
    print('Running inference...')
    result = subprocess.run(f"source {condaEnvActivateScript} && "
                   f"source {wrapperPath} {modInputPath} {modOutputPath} {weightsPath} {baseScanFile}",
                   shell=True, executable="/bin/bash",
                   capture_output=True,text=True)

    outFiles = glob.glob(os.path.join(modOutputPath, '*.nii.gz'))
    numExistingStructs = len(planC.structure)
    print(f"Importing {outFiles[0]}...")
    outputMask = sitk.ReadImage(outFiles[0])
    outputMask4M = sitk.GetArrayFromImage(outputMask)
        
    structNames = list(structToLabelMap.keys())
    for numLabel in range(len(structToLabelMap)):
        binMask = outputMask4M[:, :, :, numLabel]
        structName = 'AI_' + structNames[numLabel]
        planC = cerrStr.importStructureMask(binMask,
                                            scanNum,
                                            structName,
                                            planC,
                                            None)
    newNumStructs = len(planC.structures)
        
    structFilePath = os.path.join(outputDir, structName + '.dcm')
    structNumV = np.arange(numExistingStructs+1, newNumStructs)
    indOrigV = np.array([cerrScn.getScanNumFromUID(\
                         planC.structure[structNum].assocScanUID, planC)\
                         for structNum in structNumV], dtype=int)
    structsToExportV = structNumV[indOrigV == scanNum]
    seriesDescription = "AI Generated"
    exportOpts = {'seriesDescription': seriesDescription}
    rtstruct_iod.create(structsToExportV,
                        structFilePath,
                        planC,
                        exportOpts)
                  
os.chdir(cwd)

## Display results
Overlay AI segmentations on scan for visualization using Matplotlib

* Note: This example displays the last segmented dataset by default. Load the appropriate pyCERR archive to planC to view results for desired dataset.

In [ ]:
from cerr.viewer import showMplNb

showMplNb(planC=planC, scan_nums=scanNum,
          struct_nums=structsToExportV,
          windowCenter=-400, windowWidth=2000)